# pandas 之二：Series 的属性

---

上一篇学会了怎么"造" Series，这一篇讲怎么"看"它。

**先分清两个概念**（和 Numpy 那篇一样的道理）：

| | 写法 | 要不要括号 |
|---|---|---|
| **属性** | `s.index`、`s.shape` | **不要**括号 |
| **方法** | `s.head()`、`s.sum()` | **要**括号 |

**记忆口诀：属性是"名词"（是什么），方法是"动词"（做什么）。**

## 学习目标

1. 记住 9 个基础属性
2. 分清 `loc` / `iloc` / `at` / `iat` 这 4 个"取值工具"
3. 理解"标签索引"和"位置索引"的本质区别


In [ ]:
import pandas as pd

s1 = pd.Series([1, 2, 3, 4, 5], index=['a', 'b', 'c', 'd', 'e'], name='月份')
print(s1)


---
## 一、9 个基础属性


In [ ]:
# 1. values —— 底层数据（是一个 Numpy 数组！）
print('1. values :', s1.values, '  <- 类型是 ndarray')
print('   验证类型:', type(s1.values))

# 2. index —— 索引对象
print('\n2. index  :', s1.index)
print('   转成列表:', list(s1.index))

# 3. dtype —— 数据类型
print('\n3. dtype  :', s1.dtype)

# 4. name —— 名称
print('4. name   :', s1.name)

# 5. size —— 元素个数
print('5. size   :', s1.size)

# 6. shape —— 形状（一定是 (n,) 的一元组）
print('6. shape  :', s1.shape)

# 7. ndim —— 维度（Series 永远是 1 维）
print('7. ndim   :', s1.ndim)

# 8. empty —— 是否为空
print('8. empty  :', s1.empty)

# 9. nbytes —— 占用字节数
print('9. nbytes :', s1.nbytes)


### `values` 是 Numpy 数组 —— 这是重要的事实

这印证了前面说的：**pandas 建立在 Numpy 之上**。
所以 Numpy 里学的那些操作（切片、广播、`mean()`），
在 `s.values` 上照样能用：


In [ ]:
print('对 .values 用 Numpy 方法：')
print('  mean  =', s1.values.mean())
print('  平方  =', s1.values ** 2)
print('  布尔筛 =', s1.values[s1.values > 2])


> **反过来也成立**：Numpy 的 `np.sum(s1)` 也能直接作用在 Series 上。


In [ ]:
import numpy as np
print('np.sum(Series) =', np.sum(s1))


---
## 二、`empty`：空 Series 长什么样


In [ ]:
empty_s = pd.Series([], dtype='float64')
print('空 Series：', empty_s)
print('empty =', empty_s.empty, '  size =', empty_s.size)


注意：**`empty` 判断的是"有没有元素"，不是"元素是不是 0"**。


In [ ]:
print('值是 0 但非空：')
print(pd.Series([0, 0, 0]).empty, '  <- False，因为有 3 个元素')


---
## 三、`T` 属性：对 Series 来说是个"摆设"

Series 是一维的，转置之后还是它自己。


In [ ]:
print('s1.T 还是原样：')
print(s1.T)
print('\n两者完全一样吗？', s1.equals(s1.T))


> **原稿错在哪**：原稿在这个例子里写输出是 `Name: None, dtype: int64`。
> 但我们创建 `s1` 时给了 `name='月份'`，所以 `.T` 之后 **name 依然是 `月份`**，
> 不会变成 `None`。原稿大概是从另一个没设 name 的例子里抄过来的。


In [ ]:
print('实际 name =', s1.T.name, ' （不是 None）')


---
## 四、4 个取值工具：`loc` / `iloc` / `at` / `iat`

这是本篇**最需要记牢**的一张表：

| 工具 | 索引方式 | 取什么 | 括号里写 |
|---|---|---|---|
| `s.loc[]` | **标签**（label） | 一个或多个 | `s.loc['a']`、`s.loc['a':'c']` |
| `s.iloc[]` | **位置**（integer position） | 一个或多个 | `s.iloc[0]`、`s.iloc[0:2]` |
| `s.at[]` | **标签** | **单个**（最快） | `s.at['a']` |
| `s.iat[]` | **位置** | **单个**（最快） | `s.iat[0]` |

**`loc` 靠名字，`iloc` 靠位置，加 `at` 是只取一个的"快速版"。**


In [ ]:
print("s1.loc['a']   =", s1.loc['a'], '   (标签 a)')
print("s1.iloc[0]    =", s1.iloc[0], '   (位置 0)')

print("\ns1.loc['b':'d'] =")
print(s1.loc['b':'d'], '  <- 注意：loc 的切片【包含】末尾 d')

print('\ns1.iloc[1:3] =')
print(s1.iloc[1:3], '  <- 注意：iloc 的切片【不包含】末尾，和 Python 一样')


### ⚠️ `loc` 切片包含末尾，`iloc` 切片不包含 —— 这是最容易错的地方

| 写法 | 结果包含末尾吗 |
|---|---|---|
| `s.loc['b':'d']` | **包含** `'d'` |
| `s.iloc[1:3]` | **不包含**位置 3 |

**为什么不一样？** 因为 `loc` 是"按标签查范围"，
而标签没有"下一个"的概念，只能两端都算进去；
`iloc` 是按位置，沿用 Python 切片的习惯（含头不含尾）。

> **原稿错在哪**：原稿写 `print(s1.loc['a':"c"])` 的输出是 `1`。
> 但 `loc['a':'c']` 取的是**从 a 到 c 的三个元素**（1, 2, 3），不是单个 `1`。
> 这是把 `loc` 的单点取值和切片搞混了。


In [ ]:
print("s1.loc['a':'c'] 的真实结果：")
print(s1.loc['a':'c'])


### `at` / `iat`：只取一个值时的"加速版"


In [ ]:
print("s1.at['a']  =", s1.at['a'])
print("s1.iat[0]   =", s1.iat[0])


**为什么要有 `at` / `iat`？**
`loc` / `iloc` 要处理"可能是多个值"的复杂情况，开销大；
如果你明确只取**一个**标量，用 `at` / `iat` 更快。
在百万次循环取值的场景里，差距明显。


### 位置索引的"陷阱"：`s1[0]`

用 `[]` 直接加整数，pandas 会**先当标签查**，查不到再当位置用。
这在索引本身就是整数时（比如 `0,1,2,...`）会造成困惑。


In [ ]:
s_int_index = pd.Series([10, 20, 30])       # 索引是 0,1,2
print('索引是整数的 Series：')
print(s_int_index)
print("\ns_int_index[0] =", s_int_index[0], '  <- 这里"标签 0"和"位置 0"恰好重合，看起来正常')

# 但如果索引不是 0 开头：
s_odd_index = pd.Series([10, 20, 30], index=[5, 6, 7])
print('\n索引是 5,6,7 的 Series：')
print(s_odd_index)


这时候 `s_odd_index[0]` 会怎样？**报错**（找不到标签 0）。


In [ ]:
try:
    print(s_odd_index[0])
except KeyError as e:
    print('报 KeyError：', e, '  <- 它把 0 当【标签】去找了')

print("\n想要位置取值，必须用 iloc：s_odd_index.iloc[0] =", s_odd_index.iloc[0])


> **结论 / 最佳实践**：
> - 要**按标签**取 -> 用 `loc` / `at`
> - 要**按位置**取 -> 用 `iloc` / `iat`
> - **别用裸的 `s[x]`**，因为它"一会儿当标签、一会儿当位置"，容易出意外


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `s.shape()` | `TypeError` | 属性**不加括号** |
| `s.loc['a':'c']` 以为是单个值 | 返回 3 个元素 | `loc` 切片**两端都包含** |
| `s.iloc[1:3]` 以为包含 3 | 只到 2 | `iloc` **含头不含尾** |
| 自定义索引后用 `s[0]` | `KeyError` | 用 `.iloc[0]` |
| 以为 `.T` 会改变结构 | 其实没变 | Series 一维，转置无意义 |
| 以为 `.T` 后 name 变 None | 其实 name 保留 | 原样保留 `name` |
| `.values` 改了一个另一个也变 | 共享内存 | 要独立用 `.to_numpy().copy()` |
| 用 `empty` 判断"值是否全 0" | 语义不对 | `empty` 指"没有元素" |

---

## 六、动手练习

**练习 1**：创建 Series `s = pd.Series([10,20,30,40,50], index=list('abcde'), name='销量')`，
分别打印它的 `values`、`index`、`dtype`、`name`、`size`、`shape`、`ndim`、`empty`。

**练习 2**：用 `loc` 和 `iloc` 两种方式，取出上面 `s` 里的第 3 个元素（值 30）。

**练习 3**：先预测 `s.loc['b':'d']` 和 `s.iloc[1:3]` 的结果**各包含几个元素**，再运行验证。

**练习 4**：创建一个索引为 `[10, 20, 30]` 的 Series，
尝试用 `s[0]` 和 `s.iloc[0]` 取值，解释为什么一个报错一个正常。

**练习 5**：对 `s.values` 做一次平方运算，说明为什么可以直接用 Numpy 的方法。

---

**答案思路（先自己做）**

```python
# 1
s = pd.Series([10, 20, 30, 40, 50], index=list('abcde'), name='销量')
for attr in ['values', 'index', 'dtype', 'name', 'size', 'shape', 'ndim', 'empty']:
    print(attr, '=', getattr(s, attr))

# 2
print(s.loc['c'], s.iloc[2])

# 3
print(s.loc['b':'d'])   # 3 个：b, c, d
print(s.iloc[1:3])      # 2 个：位置 1, 2

# 4
s2 = pd.Series([1, 2, 3], index=[10, 20, 30])
# s2[0] 报 KeyError（0 不是标签）；s2.iloc[0] 正常返回 1

# 5
print(s.values ** 2)    # .values 是 ndarray，支持 Numpy 运算
```
